# Extract & Label Chess Piece Glyphs from PDF

This notebook helps you manually extract and label piece glyphs from a chess PDF book.

**Workflow:**
1. Load previous classifier (optional) to pre-filter candidates
2. Extract candidate glyphs from PDF
3. Manually label ambiguous ones
4. Train new classifier
5. Download zip with labeled glyphs + classifier

**Output:** A zip file containing:
- `glyphs/K/`, `glyphs/Q/`, etc. (labeled images)
- `classifier.pkl` (trained model for next session)

## Step 1 — Mount Google Drive and load PDF

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

In [ ]:
# ── EDIT THIS: set your PDF path ────────────────────────────────────────────
PDF_PATH = '/content/gdrive/MyDrive/chess_book.pdf'  # CHANGE THIS to your PDF

# Verify PDF exists
if not os.path.exists(PDF_PATH):
    print(f'❌ PDF not found: {PDF_PATH}')
    print(f'   Available items in /content/gdrive/MyDrive:')
    for item in os.listdir('/content/gdrive/MyDrive')[:20]:
        print(f'     - {item}')
else:
    size_mb = os.path.getsize(PDF_PATH) / (1024*1024)
    print(f'✅ PDF loaded: {PDF_PATH}  ({size_mb:.1f} MB)')

## Step 2 — Install dependencies

In [ ]:
!apt-get install -y poppler-utils
!pip install -q pdfplumber pdf2image pillow scikit-learn scikit-image

## Step 3 — Configuration

In [ ]:
import pdfplumber
from pdf2image import convert_from_path
from PIL import Image
import os
from pathlib import Path
import pickle
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from skimage import feature, transform

# Configuration
START_PAGE = 1
END_PAGE = 20
GLYPH_DPI = 150
IMG_SIZE = 32
CONFIDENCE_THRESHOLD = 0.7  # Pre-filter out low-confidence candidates

# Output directory
GLYPHS_DIR = './glyphs'
PIECE_CLASSES = ['K', 'Q', 'R', 'B', 'N']

# Create output directories
Path(GLYPHS_DIR).mkdir(exist_ok=True)
for piece in PIECE_CLASSES:
    Path(f'{GLYPHS_DIR}/{piece}').mkdir(exist_ok=True)

print(f'✅ Output directory: {GLYPHS_DIR}/')
print(f'   Subdirectories for: {" ".join(PIECE_CLASSES)}')

## Step 3.5 — Load previous classifier (optional)

In [ ]:
# If you downloaded a classifier zip from a previous session, upload it here
from google.colab import files
import zipfile

classifier = None
previous_labeled_count = {piece: 0 for piece in PIECE_CLASSES}

print('📁 Upload a previous classifier zip (optional, press Skip if none):')
print('   This will use the learned classifier to pre-filter candidates')
print()

try:
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('./previous')
            
            # Load classifier if it exists
            if os.path.exists('./previous/classifier.pkl'):
                with open('./previous/classifier.pkl', 'rb') as f:
                    classifier = pickle.load(f)
                print(f'✅ Loaded previous classifier')
            
            # Count previous labeled glyphs
            for piece in PIECE_CLASSES:
                path = f'./previous/glyphs/{piece}'
                if os.path.exists(path):
                    count = len([f for f in os.listdir(path) if f.endswith('.png')])
                    previous_labeled_count[piece] = count
            
            total_prev = sum(previous_labeled_count.values())
            print(f'✅ Found {total_prev} previously labeled glyphs')
            for piece in PIECE_CLASSES:
                if previous_labeled_count[piece] > 0:
                    print(f'   {piece}: {previous_labeled_count[piece]}')
except:
    print('ℹ️  No previous classifier uploaded (will extract all candidates)')

## Step 4 — Extract features and filter candidates

In [ ]:
def extract_image_features(img):
    """Extract simple features from glyph image."""
    # Resize to standard size
    img_resized = transform.resize(np.array(img), (IMG_SIZE, IMG_SIZE), anti_aliasing=True)
    
    features = []
    # Basic stats
    features.append(np.mean(img_resized))
    features.append(np.std(img_resized))
    
    # Aspect ratio
    features.append(img.width / max(img.height, 1))
    
    # Edge density (Sobel)
    edges = feature.sobel(img_resized)
    features.append(np.mean(edges))
    
    # Vertical and horizontal projection
    features.append(np.mean(np.sum(img_resized, axis=0)))
    features.append(np.mean(np.sum(img_resized, axis=1)))
    
    return np.array(features)

def has_non_ascii_or_special(text):
    """Check if text contains chess piece glyphs or unusual chars."""
    for char in text:
        # Only unicode chess pieces
        if char in '♔♕♖♗♘♙♚♛♜♝♞♟':
            return True
        # Non-ASCII that's not common punctuation
        if ord(char) > 127 and char not in '.,!?;:\'"()-–—…':
            return True
    return False

def render_page(pdf_path, page_num, dpi=150):
    images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1, dpi=dpi)
    return images[0] if images else None

# Find all words with glyphs
glyph_words = []
glyph_features = []

with pdfplumber.open(PDF_PATH) as pdf:
    pdf_page_count = len(pdf.pages)
    end_page = min(END_PAGE, pdf_page_count)
    
    for page_idx in range(START_PAGE - 1, end_page):
        page_num = page_idx + 1
        pdf_page = pdf.pages[page_idx]
        
        try:
            words = pdf_page.extract_words()
        except:
            continue
        
        page_image = render_page(PDF_PATH, page_idx, dpi=GLYPH_DPI)
        if page_image is None:
            continue
        
        for word in words:
            text = word.get('text', '')
            
            # Find words with glyphs
            if has_non_ascii_or_special(text):
                bbox = (word['x0'], word['top'], word['x1'], word['bottom'])
                
                # Crop and extract features
                scale = GLYPH_DPI / 72.0
                x0 = max(0, int(bbox[0] * scale))
                y0 = max(0, int(bbox[1] * scale))
                x1 = min(page_image.width, int(bbox[2] * scale))
                y1 = min(page_image.height, int(bbox[3] * scale))
                
                crop = page_image.crop((x0, y0, x1, y1))
                features = extract_image_features(crop)
                
                glyph_words.append({
                    'page': page_num,
                    'text': text,
                    'bbox': bbox,
                    'crop': crop,
                    'features': features,
                })

print(f'✅ Found {len(glyph_words)} candidate words with glyphs')

# Pre-filter with classifier if available
if classifier is not None:
    filtered_words = []
    for word in glyph_words:
        conf = np.max(classifier.predict_proba([word['features']])[0])
        word['confidence'] = conf
        if conf >= CONFIDENCE_THRESHOLD:
            filtered_words.append(word)
    
    print(f'⚡ Classifier pre-filtered to {len(filtered_words)} candidates (threshold: {CONFIDENCE_THRESHOLD})')
    glyph_words = filtered_words
else:
    print('ℹ️  No classifier available; showing all candidates')

## Step 5 — Manual labeling UI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image as IPImage, clear_output

# UI State
current_idx = 0
saved_count = {piece: previous_labeled_count[piece] for piece in PIECE_CLASSES}
skipped = 0
discarded = 0
labeled_images = {piece: [] for piece in PIECE_CLASSES}

def show_glyph(idx):
    """Display a glyph and show labeling buttons."""
    global current_idx, saved_count, skipped, discarded
    
    if idx >= len(glyph_words):
        clear_output()
        print(f'✅ Labeling complete!')
        print(f'\nSaved:')
        for piece in PIECE_CLASSES:
            total = saved_count[piece]
            print(f'  {piece}: {total}')
        print(f'  Skipped: {skipped}')
        print(f'  Discarded: {discarded}')
        return
    
    current_idx = idx
    word_info = glyph_words[idx]
    crop = word_info['crop']
    
    # Display
    clear_output()
    conf_text = f" (confidence: {word_info.get('confidence', 0):.2%})" if 'confidence' in word_info else ""
    print(f'Glyph {idx + 1}/{len(glyph_words)} — P{word_info["page"]}: "{word_info["text"]}"{conf_text}')
    print()
    display(crop)
    print()
    
    # Buttons
    def save_glyph(piece):
        count = saved_count[piece]
        filename = f'{GLYPHS_DIR}/{piece}/{count + 1:04d}.png'
        crop.save(filename)
        saved_count[piece] += 1
        labeled_images[piece].append(word_info['features'])
        show_glyph(idx + 1)
    
    def skip():
        global skipped
        skipped += 1
        show_glyph(idx + 1)
    
    def discard():
        global discarded
        discarded += 1
        show_glyph(idx + 1)
    
    buttons = [
        widgets.Button(description='K (King)', button_style='info'),
        widgets.Button(description='Q (Queen)', button_style='info'),
        widgets.Button(description='R (Rook)', button_style='info'),
        widgets.Button(description='B (Bishop)', button_style='info'),
        widgets.Button(description='N (Knight)', button_style='info'),
        widgets.Button(description='Skip', button_style='warning'),
        widgets.Button(description='❌ Discard', button_style='danger'),
    ]
    
    buttons[0].on_click(lambda _: save_glyph('K'))
    buttons[1].on_click(lambda _: save_glyph('Q'))
    buttons[2].on_click(lambda _: save_glyph('R'))
    buttons[3].on_click(lambda _: save_glyph('B'))
    buttons[4].on_click(lambda _: save_glyph('N'))
    buttons[5].on_click(lambda _: skip())
    buttons[6].on_click(lambda _: discard())
    
    display(widgets.HBox(buttons))

# Start
if len(glyph_words) > 0:
    show_glyph(0)
else:
    print('❌ No candidates to label')

## Step 6 — Train classifier

In [ ]:
# Collect all labeled data
X_train = []
y_train = []

# Load newly labeled glyphs from disk
for piece_idx, piece in enumerate(PIECE_CLASSES):
    path = f'{GLYPHS_DIR}/{piece}'
    if os.path.exists(path):
        for img_file in os.listdir(path):
            if img_file.endswith('.png'):
                img = Image.open(f'{path}/{img_file}').convert('L')
                features = extract_image_features(img)
                X_train.append(features)
                y_train.append(piece_idx)

if len(X_train) > 10:  # Need at least some samples
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    
    # Train classifier
    new_classifier = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=10)
    new_classifier.fit(X_train, y_train)
    
    # Save classifier
    with open(f'{GLYPHS_DIR}/classifier.pkl', 'wb') as f:
        pickle.dump(new_classifier, f)
    
    print(f'✅ Trained classifier on {len(X_train)} labeled glyphs')
    print(f'   Accuracy: {new_classifier.score(X_train, y_train):.1%}')
    print(f'\n   Model saved to glyphs/classifier.pkl')
else:
    print('⚠️  Not enough labeled samples to train classifier (need ≥10)')

## Step 7 — Export zip with classifier

In [ ]:
import shutil

# Create zip file
zip_filename = 'chess_glyphs_classifier.zip'
shutil.make_archive('chess_glyphs_classifier', 'zip', '.', GLYPHS_DIR)

# Count final glyphs
final_counts = {}
total_glyphs = 0
for piece in PIECE_CLASSES:
    path = f'{GLYPHS_DIR}/{piece}'
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith('.png')])
        final_counts[piece] = count
        total_glyphs += count

print(f'✅ Export complete!')
print(f'\n📦 Zip file: {zip_filename}')
print(f'\nContents:')
print(f'  glyphs/classifier.pkl (trained model)')
for piece in PIECE_CLASSES:
    count = final_counts.get(piece, 0)
    print(f'  glyphs/{piece}/ ({count} images)')
print(f'\n  TOTAL: {total_glyphs} labeled glyphs')
print(f'\n💾 Download {zip_filename} and upload to this notebook in the next session')
print(f'   The classifier will improve with each round of labeling!')

# Show download link
from google.colab import files
files.download(zip_filename)